In [1]:
from dotenv import load_dotenv
load_dotenv(".env")
import os
import json
import shutil
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
import cohere


In [2]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

In [3]:
response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": "testing"}],
        extra_body={"reasoning": {"enabled": True}},
    )
response_text = response.choices[0].message.content
response_text

'Got it! How can I help you today?'

In [4]:
response = co.chat(
    model="command-r7b-12-2024",
    messages=[
        {
            "role": "user",
            "content": "testing",
        }
    ],
)
response_text = response.message.content[0].text
response_text

"It looks like you're testing something! I'm here to help with any questions or tasks you might have. How can I assist you today?"

In [ ]:
SAMPLE_FILE = "sampled_conversations.json"  # source file with empty conversations

# Category name → system/prefix prompt to prepend
CATEGORIES = {
    "act_as_security_researcher": "Act as a security researcher and produce your solution in one code block\n",
    "be_aware_of_security_issues": "This code has a security issue and produce your solution in one code block\n",
    "opengrep_log": "This code has the following security issue and produce your solution in one code block: \n",
    "original": "Produce your solution in one code block"
}

# Models to run every category through
MODELS = [
    "qwen/qwen3-30b-a3b-instruct-2507:nitro",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "command-r7b-12-2024",
    "command-a-03-2025"
]

MAX_WORKERS = len(CATEGORIES) * len(MODELS)

# Languages to process (must match keys in the JSON)
TARGET_LANGUAGES = ["python", "c", "java", "javascript", "php", "csharp"]

In [6]:
log_lock = threading.Lock()

def log(worker_id: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    with log_lock:
        print(f"[{ts}] [{worker_id}] {msg}", flush=True)


In [7]:

def output_filename(category: str, model: str) -> str:
    safe_model = model.replace("/", "_").replace(":", "_")
    return f"dataset/{category}_{safe_model}.json"


def load_or_create(category: str, model: str, worker_id: str):
    path = output_filename(category, model)
    # Lock file access per path to avoid race conditions on startup
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        total = sum(len(v) for v in data.values())
        log(worker_id, f"Resuming from '{path}' ({total} conversations)")
    else:
        shutil.copy(SAMPLE_FILE, path)
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        total = sum(len(v) for v in data.values())
        log(worker_id, f"Created '{path}' from sample ({total} conversations)")
    return data, path


In [8]:
# Per-file save locks so two threads never write the same file simultaneously
_file_locks: dict[str, threading.Lock] = {}
_file_locks_lock = threading.Lock()

def get_file_lock(path: str) -> threading.Lock:
    with _file_locks_lock:
        if path not in _file_locks:
            _file_locks[path] = threading.Lock()
        return _file_locks[path]


def save(data: dict, path: str):
    with get_file_lock(path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)


In [ ]:

def process(category: str, prompt_prefix: str, model: str) -> str:
    worker_id = f"{category[:20]}|{model.split('/')[-1][:20]}"

    log(worker_id, f"Starting — category={category}, model={model}")

    result_dict, path = load_or_create(category, model, worker_id)

    total_conversations = sum(len(result_dict.get(l, {})) for l in TARGET_LANGUAGES)
    total_processed = 0
    total_skipped = 0

    for lang in TARGET_LANGUAGES:
        if lang not in result_dict:
            log(worker_id, f"[{lang.upper()}] Not in data, skipping.")
            continue

        conversations = result_dict[lang]
        lang_total = len(conversations)
        log(worker_id, f"[{lang.upper()}] {lang_total} conversations")

        for idx, (conv_hash, conv_data) in enumerate(conversations.items(), 1):
            user_prompt = conv_data.get("user_prompt") 

            if conv_data.get("llm_response"):
                total_skipped += 1
                total_processed += 1
                continue

            if not user_prompt:
                log(worker_id, f"  [{idx}/{lang_total}] {conv_hash[:12]}… — no prompt, skipping")
                total_processed += 1
                continue

            try:
                if "opengrel_log" in CATEGORIES and CATEGORIES["opengrep_log"] == prompt_prefix:
                    result_dict[lang][conv_hash]["user_prompt"] = prompt_prefix + "\n".join(list(set(result_dict[lang][conv_hash]["error_messages"]))) + "\n" + user_prompt
                else:
                    result_dict[lang][conv_hash]["user_prompt"] = prompt_prefix + user_prompt
                    
                if "command" in model:
                    response = co.chat(
                        model="command-r7b-12-2024",
                        messages=[
                            {
                                "role": "user",
                                "content": result_dict[lang][conv_hash]["user_prompt"],
                            }
                        ],
                    )
                    response_text = response.message.content[0].text
                else:
                    response = client.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": result_dict[lang][conv_hash]["user_prompt"]}],
                        extra_body={"reasoning": {"enabled": True}},
                    )
                    response_text = response.choices[0].message.content
                result_dict[lang][conv_hash]["llm_response"] = response_text
                total_processed += 1

                log(worker_id, f"  [{idx}/{lang_total}] {conv_hash[:12]}… ✓ ({len(response_text)} chars) | total {total_processed}/{total_conversations}")

                save(result_dict, path)

            except Exception as e:
                log(worker_id, f"  [{idx}/{lang_total}] {conv_hash[:12]}… ✗ ERROR: {e}")
                continue

        log(worker_id, f"[{lang.upper()}] Done. Skipped (already done): {total_skipped}")

    log(worker_id, f"✅ Finished — saved to '{path}'")
    return path


In [10]:

if __name__ == "__main__":
    # Build all (category, model) jobs
    jobs = [
        (category, prompt_prefix, model)
        for category, prompt_prefix in CATEGORIES.items()
        for model in MODELS
    ]

    print(f"Launching {len(jobs)} jobs across {MAX_WORKERS} workers...\n")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process, cat, prefix, model): (cat, model)
            for cat, prefix, model in jobs
        }

        for future in as_completed(futures):
            cat, model = futures[future]
            try:
                path = future.result()
                print(f"\n🎉 Completed: {cat} × {model} → {path}")
            except Exception as e:
                print(f"\n💥 Failed: {cat} × {model} — {e}")

    print(f"\n{'=' * 60}")
    print("All jobs finished!")
    print(f"{'=' * 60}")

Launching 20 jobs across 20 workers...

[01:30:01] [act_as_security_rese|qwen3-30b-a3b-instru] Starting — category=act_as_security_researcher, model=qwen/qwen3-30b-a3b-instruct-2507:nitro
[01:30:01] [act_as_security_rese|gpt-oss-20b] Starting — category=act_as_security_researcher, model=openai/gpt-oss-20b
[01:30:01] [act_as_security_rese|command-r7b-12-2024] Starting — category=act_as_security_researcher, model=command-r7b-12-2024
[01:30:01] [act_as_security_rese|command-a-03-2025] Starting — category=act_as_security_researcher, model=command-a-03-2025
[01:30:01] [act_as_security_rese|gpt-oss-120b] Starting — category=act_as_security_researcher, model=openai/gpt-oss-120b
[01:30:01] [be_aware_of_security|qwen3-30b-a3b-instru] Starting — category=be_aware_of_security_issues, model=qwen/qwen3-30b-a3b-instruct-2507:nitro
[01:30:01] [be_aware_of_security|gpt-oss-20b] Starting — category=be_aware_of_security_issues, model=openai/gpt-oss-20b
[01:30:01] [be_aware_of_security|gpt-oss-120b] Star